In [43]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../data/raw/chembl_37.db')

In [44]:
# Origin of non-exact (!=) values
pd.read_sql_query('''
SELECT s.src_description, ac.standard_relation, count(*) AS "Count of IC50 Measurements"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND (ac.data_validity_comment IS NULL OR ac.data_validity_comment != 'Potential transcription error')
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id != 'CHEMBL3220211'
GROUP BY s.src_description, ac.standard_relation;
''', con)

,src_description,standard_relation,Count of IC50 Measurements
0,BindingDB Patent Bioactivity Data,<,4053
1,BindingDB Patent Bioactivity Data,=,5470
2,BindingDB Patent Bioactivity Data,>,239
3,Chemical Probe data from Scientific Literature,=,12
4,Chemical Probe data from Scientific Literature,>,7
5,EUbOPEN Chemogenomic Library,=,9
6,EUbOPEN Chemogenomic Library Literature Data,=,6
7,EUbOPEN Chemogenomic Library Literature Data,>,8
8,SGC Frankfurt - Donated Chemical Probes,=,24
9,Scientific Literature,<,56


In [45]:
# Range of IC50 values classified 'outside typical range' (Data Validity Comment)
pd.read_sql_query('''
SELECT ac.data_validity_comment, max(ac.standard_value) AS "Max IC50 Value", min(ac.standard_value) AS "Min IC50 Value", count(*) AS "Count of IC50 Measurements"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_relation = '='
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id != 'CHEMBL3220211'
AND s.src_description IN ('BindingDB Patent Bioactivity Data','Scientific Literature')
AND NOT (s.src_description = 'BindingDB Patent Bioactivity Data' AND ac.standard_value IN (50500,550,5500,2750))
GROUP BY ac.data_validity_comment;
''', con)


,data_validity_comment,Max IC50 Value,Min IC50 Value,Count of IC50 Measurements
0,NaN,6880000,0.1,9261
1,Outside typical range,280000000,102000.0,37


In [46]:
# (−)-JQ1 (CHEMBL3220211) excluded due to contradictory values
pd.read_sql_query('''
SELECT m.chembl_id, max(ac.standard_value) AS "Max IC50 Value", min(ac.standard_value) AS "Min IC50 Value", count(*) AS "Count of IC50 Measurements"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_relation = '='
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND (ac.data_validity_comment IS NULL OR ac.data_validity_comment != 'Potential transcription error')
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id = 'CHEMBL3220211'
AND s.src_description IN ('BindingDB Patent Bioactivity Data','Scientific Literature')
AND NOT (s.src_description = 'BindingDB Patent Bioactivity Data' AND ac.standard_value IN (50500,550,5500,2750));
''', con)


,chembl_id,Max IC50 Value,Min IC50 Value,Count of IC50 Measurements
0,CHEMBL3220211,52120000,29,8


In [47]:
# Correction for expression hosts Sf9, BL21, CHO
pd.read_sql_query('''
SELECT bao.label, a.assay_cell_type, count(*) AS "Count of IC50 Measurements"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
LEFT JOIN bioassay_ontology bao ON bao.bao_id = a.bao_format
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_relation = '='
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND (ac.data_validity_comment IS NULL OR ac.data_validity_comment != 'Potential transcription error')
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id != 'CHEMBL3220211'
AND s.src_description IN ('BindingDB Patent Bioactivity Data','Scientific Literature')
AND NOT (s.src_description = 'BindingDB Patent Bioactivity Data' AND ac.standard_value IN (50500,550,5500,2750))
GROUP BY bao.label, a.assay_cell_type;
''', con)

,label,assay_cell_type,Count of IC50 Measurements
0,assay format,NaN,1283
1,cell-based format,NaN,591
2,cell-based format,231MFP,1
3,cell-based format,BL21-DE3,11
4,cell-based format,CHO,18
5,cell-based format,HCT-116,6
6,cell-based format,HEK-293T,2
7,cell-based format,HEK293,30
8,cell-based format,HEK293T,1
9,cell-based format,HEL,2


In [48]:
# Most frequent IC50 values, showing the accumulation at 50,500 nM, 550 nM, 5,500 nM, and 2,750 nM
pd.read_sql_query('''
SELECT ac.standard_value, s.src_description, count(*) AS "Count of IC50 Measurements", count(DISTINCT a.doc_id) AS "Count of Source Documents"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_relation = '='
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND (ac.data_validity_comment IS NULL OR ac.data_validity_comment != 'Potential transcription error')
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id != 'CHEMBL3220211'
AND s.src_description IN ('BindingDB Patent Bioactivity Data','Scientific Literature')
GROUP BY ac.standard_value, s.src_description ORDER BY count(*) DESC LIMIT 10;
''', con)


,standard_value,src_description,Count of IC50 Measurements,Count of Source Documents
0,50500.00,BindingDB Patent Bioactivity Data,754,1
1,550.00,BindingDB Patent Bioactivity Data,436,11
2,5500.00,BindingDB Patent Bioactivity Data,208,7
3,2750.00,BindingDB Patent Bioactivity Data,148,6
4,15848.93,Scientific Literature,56,21
5,1258.93,Scientific Literature,55,20
6,501.19,Scientific Literature,54,19
7,19952.62,Scientific Literature,54,20
8,12589.25,Scientific Literature,53,23
9,630.96,Scientific Literature,52,22


In [49]:
# Most frequent IC50 values in patent data, verifying the mid-point accumulation
pd.read_sql_query('''
SELECT ac.standard_value, s.src_description, count(*) AS "Count of IC50 Measurements", count(DISTINCT a.doc_id) AS "Count of Source Documents"
FROM activities ac
JOIN assays a ON a.assay_id = ac.assay_id 
JOIN molecule_dictionary m ON m.molregno = ac.molregno
JOIN source s ON s.src_id = a.src_id
WHERE a.tid = 103454 
AND ac.standard_type = 'IC50'
AND ac.standard_relation = '='
AND ac.standard_value IS NOT NULL
AND ac.standard_units = 'nM'
AND (ac.data_validity_comment IS NULL OR ac.data_validity_comment != 'Potential transcription error')
AND ac.potential_duplicate = 0
AND a.assay_type = 'B'
AND m.chembl_id != 'CHEMBL3220211'
AND s.src_description IN ('BindingDB Patent Bioactivity Data')
GROUP BY ac.standard_value, s.src_description ORDER BY count(*) DESC LIMIT 10;
''', con)


,standard_value,src_description,Count of IC50 Measurements,Count of Source Documents
0,50500,BindingDB Patent Bioactivity Data,754,1
1,550,BindingDB Patent Bioactivity Data,436,11
2,5500,BindingDB Patent Bioactivity Data,208,7
3,2750,BindingDB Patent Bioactivity Data,148,6
4,12,BindingDB Patent Bioactivity Data,40,13
5,18,BindingDB Patent Bioactivity Data,39,13
6,16,BindingDB Patent Bioactivity Data,38,10
7,17,BindingDB Patent Bioactivity Data,38,14
8,2,BindingDB Patent Bioactivity Data,36,6
9,13,BindingDB Patent Bioactivity Data,36,9


In [50]:
con.close()